# SQL — Intermediate, Session 01: CTEs / Window Functions / Correlated Subqueries / Self-Joins

Dataset: `datasets/chinook.db`. First intermediate notebook — beginner SQL is fully done.

Select the **"Python (SCLT tutor venv)"** kernel before running anything.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../../../datasets/chinook.db")

def run(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

---
## Q1 — CTE

Write a CTE named `CustomerSpend` that computes, from `Invoice`, `CustomerId` and
`TotalSpent` (`SUM(Total)`), grouped by `CustomerId`. Then, in the main query, select
from that CTE only rows where `TotalSpent > 40`, returning `CustomerId` and `TotalSpent`,
ordered by `TotalSpent` descending.

In [3]:
query = """
-- your SQL here
WITH CustomerSpend AS(
    SELECT 
        CustomerId,
        sum(Total) AS TotalSpent
    FROM
        Invoice
    GROUP BY
        CustomerId
)
SELECT
    *
FROM
    CustomerSpend
WHERE
    TotalSpent > 40
ORDER BY
    TotalSpent DESC
"""
run(query)

,CustomerId,TotalSpent
0,6,49.62
1,26,47.62
2,57,46.62
3,45,45.62
4,46,45.62
5,24,43.62
6,28,43.62
7,37,43.62
8,7,42.62
9,25,42.62


---
## Q2 — Window function: RANK() with PARTITION BY

From `Track`, for rows where `AlbumId = 1`, return `AlbumId`, `Name`, `Milliseconds`,
and a `LengthRank` column computed as `RANK() OVER (PARTITION BY AlbumId ORDER BY
Milliseconds DESC)`. Order the final result by `LengthRank` ascending.

In [4]:
query = """
-- your SQL here
WITH r AS(
    SELECT 
        AlbumId,
        Name,
        Milliseconds,
        RANK() OVER (PARTITION BY AlbumId ORDER BY Milliseconds DESC) AS LengthRank
    FROM 
        Track
    WHERE
        AlbumId = 1
)
SELECT 
    *
FROM
    r
ORDER BY
    LengthRank ASC
"""
run(query)

,AlbumId,Name,Milliseconds,LengthRank
0,1,For Those About To Rock (We Salute You),343719,1
1,1,Spellbound,270863,2
2,1,Evil Walks,263497,3
3,1,Breaking The Rules,263288,4
4,1,Let's Get It Up,233926,5
5,1,Inject The Venom,210834,6
6,1,Night Of The Long Knives,205688,7
7,1,Put The Finger On You,205662,8
8,1,Snowballed,203102,9
9,1,C.O.D.,199836,10


---
## Q3 — Window function: ROW_NUMBER() + CTE, top-N per group

Using a CTE, compute for every track a `ROW_NUMBER() OVER (PARTITION BY GenreId ORDER BY
Milliseconds DESC)` aliased `rn`, keeping `GenreId`, `Name`, `Milliseconds`. In the outer
query, return only rows where `rn <= 3` **and** `GenreId = 1` (i.e. the 3 longest tracks
in genre 1) — columns `GenreId`, `Name`, `Milliseconds`.

In [14]:
query = """
-- your SQL here
WITH rownum AS(
    SELECT
        GenreId,
        Name,
        Milliseconds,
        ROW_NUMBER() OVER (PARTITION BY GenreId ORDER BY Milliseconds DESC) AS rn
    FROM
        Track
)
SELECT
    GenreId,
    Name,
    Milliseconds
FROM
    rownum
WHERE
    rn <= 3
AND GenreID = 1
"""
run(query)

,GenreId,Name,Milliseconds
0,1,Dazed And Confused,1612329
1,1,Space Truckin',1196094
2,1,Dazed And Confused,1116734


---
## Q4 — Correlated-style subquery: above-average aggregate

From `Invoice`, group by `CustomerId` to get `TotalSpent` (`SUM(Total)`). Keep only
customers whose `TotalSpent` is greater than the **average `TotalSpent` across all
customers** — compute that average with a subquery over the same per-customer sums
(i.e. `SELECT AVG(TotalSpent) FROM (SELECT SUM(Total) AS TotalSpent FROM Invoice GROUP BY
CustomerId)`). Return `CustomerId`, `TotalSpent`, ordered by `TotalSpent` descending,
limit 10.

In [18]:
query = """
-- your SQL here
SELECT
    CustomerId,
    sum(Total) AS TotalSpent
FROM
    Invoice
GROUP BY
    CustomerId
HAVING
    sum(Total) >(SELECT AVG(TotalSpent) FROM (SELECT sum(Total) AS TotalSpent FROM Invoice GROUP BY CustomerId) AS subquery)
ORDER BY 
    sum(Total) DESC
LIMIT 10
"""
run(query)

,CustomerId,TotalSpent
0,6,49.62
1,26,47.62
2,57,46.62
3,45,45.62
4,46,45.62
5,24,43.62
6,28,43.62
7,37,43.62
8,7,42.62
9,25,42.62


---
## Q5 — Self-join

The `Employee` table has a `ReportsTo` column that holds another employee's
`EmployeeId` (their manager) — or `NULL` if they have no manager. `LEFT JOIN`
`Employee` to itself (alias the left copy `e`, the right copy `m`) on
`e.ReportsTo = m.EmployeeId`. Return `e.EmployeeId`, `e.FirstName || ' ' ||
e.LastName` as `EmployeeName`, and `m.FirstName || ' ' || m.LastName` as
`ManagerName`, ordered by `e.EmployeeId`.

In [13]:
query = """
-- your SQL here
SELECT
    e.EmployeeId,
    e.FirstName || ' ' || e.LastName AS EmployeeName,
    m.FirstName || ' ' || m.LastName AS ManagerName
FROM
    Employee e
LEFT JOIN
    Employee m
ON
    e.ReportsTo = m.EmployeeId
ORDER BY
    e.EmployeeId
"""
run(query)

,EmployeeId,EmployeeName,ManagerName
0,1,Andrew Adams,NaN
1,2,Nancy Edwards,Andrew Adams
2,3,Jane Peacock,Nancy Edwards
3,4,Margaret Park,Nancy Edwards
4,5,Steve Johnson,Nancy Edwards
5,6,Michael Mitchell,Andrew Adams
6,7,Robert King,Michael Mitchell
7,8,Laura Callahan,Michael Mitchell


---
## Done?

Say "done" in chat once all 5 run and look right. This is intermediate SQL notebook 1 of
several — more window functions and set operations (UNION, INTERSECT) are next.